# Find the best delta_max for each dataset (label for meta learning) from previous learning results

# For each dataset, which XE(delta_max) is the best? (4 deltas): Choosing Best CV configuration and report test results

In [ ]:
top = 1                  # how many top results do you want? we use top 1 result
j_list = [13, 16, 17]   # 9:Acc, 10:Pre, 11:Rec, 12:Spe, 13:F1, 14:GM, 15:BA, 16:ROCAUC, 17:PRAUC
metrics = ['Acc', 'Pre', 'Rec', 'Spe', 'F1', 'GM', 'BA', 'ROCAUC', 'PRAUC']

In [ ]:
Best_Deltas = pd.DataFrame(final_list, columns=['DATA'])
print("TOP {} Average".format(top))
for j in j_list:
    print("METRIC:", metrics[j-9])
    # XE (deltas)
    df = pd.read_csv('./Meta Results/XE(deltas)_result.csv', low_memory=False)
    new_col = ['Unnamed: 0', 'Dataset'] + [col for col in df.columns if "_LR" in col or "_LG" in col]
    df = df[new_col]
    df.rename(columns={'Unnamed: 0':'Metrics'}, inplace=True)
    base_ind = 2
    num_class = (df.shape[1]-base_ind)/(5+5+5+5)  # X with 4 deltas
    meth1_ind = int(base_ind+num_class*5)
    meth2_ind = int(base_ind+num_class*10)
    meth3_ind = int(base_ind+num_class*15)
    metrics = ['Acc', 'Pre', 'Rec', 'Spe', 'F1', 'GM', 'BA', 'ROCAUC', 'PRAUC']  
    df_res = pd.DataFrame({'DATA':[0 for i in range(top*len(final_list))],
                           'XE(1.5)_tr':[0 for i in range(top*len(final_list))],
                           'XE(1.5)_va':[0 for i in range(top*len(final_list))],
                           'XE(1.5)_t':[0 for i in range(top*len(final_list))],
                           'XE(2.0)_tr':[0 for i in range(top*len(final_list))],
                           'XE(2.0)_va':[0 for i in range(top*len(final_list))],
                           'XE(2.0)_t':[0 for i in range(top*len(final_list))],
                           'XE(2.5)_tr':[0 for i in range(top*len(final_list))],
                           'XE(2.5)_va':[0 for i in range(top*len(final_list))],
                           'XE(2.5)_t':[0 for i in range(top*len(final_list))],
                           'XE(3.0)_tr':[0 for i in range(top*len(final_list))],
                           'XE(3.0)_va':[0 for i in range(top*len(final_list))],
                           'XE(3.0)_t':[0 for i in range(top*len(final_list))]})

    for i in range(len(final_list)):        # in one dataset
        df_i = df[df["Dataset"] == str(final_list[i])]   # i_th dataset
        df_i_base = df_i.iloc[:,:base_ind]               # i_th dataset base
        df_i_meth1 = df_i.iloc[:,base_ind:meth1_ind]     # i_th dataset XEL results (163 classifier X 5 Resam strategy)
        df_i_meth2 = df_i.iloc[:,meth1_ind:meth2_ind]    # i_th dataset XEL results (163 classifier X 5 Resam strategy)
        df_i_meth3 = df_i.iloc[:,meth2_ind:meth3_ind]    # i_th dataset XEL results (163 classifier X 5 Resam strategy)
        df_i_meth4 = df_i.iloc[:,meth3_ind:]             # i_th dataset XEL results (163 classifier X 5 Resam strategy)
        best_meth1 = pd.DataFrame(df_i_meth1.iloc[j,:]).iloc[:,0].sort_values(ascending=False)[:top].index
        df_i_meth1_best = df_i_meth1.loc[:,best_meth1] 
        best_meth2 = pd.DataFrame(df_i_meth2.iloc[j,:]).iloc[:,0].sort_values(ascending=False)[:top].index
        df_i_meth2_best = df_i_meth2.loc[:,best_meth2]
        best_meth3 = pd.DataFrame(df_i_meth3.iloc[j,:]).iloc[:,0].sort_values(ascending=False)[:top].index
        df_i_meth3_best = df_i_meth3.loc[:,best_meth3]
        best_meth4 = pd.DataFrame(df_i_meth4.iloc[j,:]).iloc[:,0].sort_values(ascending=False)[:top].index
        df_i_meth4_best = df_i_meth4.loc[:,best_meth4]
        df_i_best = pd.concat([df_i_base,df_i_meth1_best],axis=1)
        df_i_best = pd.concat([df_i_best,df_i_meth2_best],axis=1)
        df_i_best = pd.concat([df_i_best,df_i_meth3_best],axis=1)
        df_i_best = pd.concat([df_i_best,df_i_meth4_best],axis=1)
        for k in range(top):
            df_res.iloc[i*top+k:i*top+k+1,0] = final_list[i]
            df_res.iloc[i*top+k:i*top+k+1,1] = list(df_i_best.iloc[[j-9],2+k])
            df_res.iloc[i*top+k:i*top+k+1,2] = list(df_i_best.iloc[[j],2+k])
            df_res.iloc[i*top+k:i*top+k+1,3] = list(df_i_best.iloc[[j+9],2+k])
            df_res.iloc[i*top+k:i*top+k+1,4] = list(df_i_best.iloc[[j-9],2+top+k])
            df_res.iloc[i*top+k:i*top+k+1,5] = list(df_i_best.iloc[[j],2+top+k])
            df_res.iloc[i*top+k:i*top+k+1,6] = list(df_i_best.iloc[[j+9],2+top+k])
            df_res.iloc[i*top+k:i*top+k+1,7] = list(df_i_best.iloc[[j-9],2+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,8] = list(df_i_best.iloc[[j],2+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,9] = list(df_i_best.iloc[[j+9],2+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,10] = list(df_i_best.iloc[[j-9],2+top+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,11] = list(df_i_best.iloc[[j],2+top+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,12] = list(df_i_best.iloc[[j+9],2+top+top+top+k])
            
    vt_col = [col for col in df_res.columns if not "_tr" in col]
    df_va_t = df_res[vt_col]

    df_average_xe= pd.DataFrame({'DATA':[final_list[i] for i in range(len(final_list))]})
    for col in df_va_t.columns[1:]:
        df_average_xe[col] = [0 for i in range(len(final_list))]
    for i in range(len(final_list)):        # in one dataset
        df_i = df_va_t[df_va_t["DATA"] == final_list[i]]   # i_th dataset
        for col in df_va_t.columns[1:]:
            df_average_xe.loc[i,col] = np.average(df_i.loc[top*i:top*(i+1),col].astype(float))  
            
    v_col = [col for col in vt_col if not "_t" in col]
    t_col = [col for col in vt_col if not "_va" in col]
    df_val = df_res[v_col]
    df_test = df_res[t_col]
    
    list_best = []
    for i, row in df_val.iterrows():
        row_best = np.max(row[1:])
        for col in df_val.columns[1:]:
            if df_val.loc[i, col] == row_best:
                delta = col.strip('XE(').strip(')_va')
                list_best.append(delta)
                break                                       # IF there are many same (best) ones, the earlier one is selected
    df_val['Best_Delta'] = list_best
    # Save Ground Truth (Optimal delta_max based on CV results)
    Best_Deltas[f'{metrics[j-9]}'] = list_best
    
    list_best = []
    for i, row in df_test.iterrows():
        row_best = np.max(row[1:])
        for col in df_test.columns[1:]:
            if df_test.loc[i, col] == row_best:
                delta = col.strip('XE(').strip(')_t')
                list_best.append(delta)
                break                                       # IF there are many same (best) ones, the earlier one is selected
    df_test['Best_Delta'] = list_best
    # Save the test scores 
    df_test.to_csv(f'./Meta Results/test_scores_from_102data_{metrics[j-9]}.csv', index=False)            
            
    test_result = ['DATA']
    for col in df_res.columns:
        if col[-2:] == '_t':
            test_result.append(col)
    df_test = df_res[test_result]
    
    df_average_xe= pd.DataFrame({'DATA':[final_list[i] for i in range(len(final_list))]})
    for col in df_test.columns[1:]:
        df_average_xe[col] = [0 for i in range(len(final_list))]
    for i in range(len(final_list)):        # in one dataset
        df_i = df_test[df_test["DATA"] == final_list[i]]   # i_th dataset
        for col in df_test.columns[1:]:
            df_average_xe.loc[i,col] = np.average(df_i.loc[top*i:top*(i+1),col].astype(float))       
    
    list_best = []
    for i, row in df_average_xe.iterrows():
        row_best = np.max(row[1:])
        for col in df_average_xe.columns[1:]:
            if df_average_xe.loc[i, col] == row_best:
                delta = col.strip('XE(').strip(')_t')
                list_best.append(delta)
                break                                       # IF there are many same (best) ones, the earlier one is selected
    df_average_xe['Best_Delta'] = list_best
    print(df_average_xe)
    Best_Deltas[f'{metrics[j-9]}'] = list_best
Best_Deltas.to_csv('./Meta Results/Best_Deltas(test).csv', index=False)